<h1>1. Install packages and check GPU </h1>

In [2]:
# ============================================================
# 1. ENVIRONMENT + GPU CHECK
# ============================================================

!pip install -q gdown opencv-python-headless pandas scikit-learn tqdm

import os
import gc
import time
import random
import tarfile
import shutil
import math
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

    print("CUDA version:", torch.version.cuda)

    # GPU performance settings
    torch.backends.cudnn.benchmark = True

    # TensorFloat-32 can improve supported CUDA matrix operations
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    device = torch.device("cuda")

else:

    raise RuntimeError(
        "CUDA GPU was not detected. "
        "Please enable Runtime > Change runtime type > GPU."
    )

print("\nUsing device:", device)

ENVIRONMENT
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
CUDA version: 13.0

Using device: cuda


<h1>2. Mount Google Drive </h1>

In [3]:
# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

from google.colab import drive

MOUNT_POINT = "/content/gdrive"

# Mount at a clean path to avoid /content/drive conflicts
drive.mount(MOUNT_POINT)

print("\nGoogle Drive mounted successfully.")

print(
    "MyDrive exists:",
    os.path.exists(
        f"{MOUNT_POINT}/MyDrive"
    )
)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

Google Drive mounted successfully.
MyDrive exists: True


<h1>3. Create project directories </h1>

In [4]:
# ============================================================
# 3. PROJECT DIRECTORIES
# ============================================================

BASE_DIR = "/content/Future-Face"

ARCHIVE_DIR = os.path.join(
    BASE_DIR,
    "archives"
)

DATA_DIR = os.path.join(
    BASE_DIR,
    "UTKFace"
)

CHECKPOINT_DIR = os.path.join(
    MOUNT_POINT,
    "MyDrive",
    "Colab Notebooks",
    "Dataset",
    "Future-Face",
    "checkpoints"
)

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("BASE:", BASE_DIR)
print("ARCHIVES:", ARCHIVE_DIR)
print("DATA:", DATA_DIR)
print("CHECKPOINTS:", CHECKPOINT_DIR)

BASE: /content/Future-Face
ARCHIVES: /content/Future-Face/archives
DATA: /content/Future-Face/UTKFace
CHECKPOINTS: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints


<h1>4. Download all 3 UTKFace archives </h1>

In [5]:
# ============================================================
# 4. DOWNLOAD UTKFACE PART 1, 2, 3
# ============================================================

import gdown

FILES = {
    "part1.tar.gz":
        "1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW",

    "part2.tar.gz":
        "19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b",

    "part3.tar.gz":
        "1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b"
}

for filename, file_id in FILES.items():

    output_path = os.path.join(
        ARCHIVE_DIR,
        filename
    )

    if os.path.exists(output_path):

        size_mb = (
            os.path.getsize(output_path)
            / 1024**2
        )

        print(
            f"Already exists: {filename} "
            f"({size_mb:.1f} MB)"
        )

        continue

    print("\n" + "=" * 70)
    print("Downloading:", filename)
    print("=" * 70)

    url = (
        f"https://drive.google.com/uc?id={file_id}"
    )

    gdown.download(
        url,
        output_path,
        quiet=False
    )

print("\nAll archives downloaded.")


Downloading: part1.tar.gz


Downloading...
From (original): https://drive.google.com/uc?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW
From (redirected): https://drive.google.com/uc?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&confirm=t&uuid=12a28c27-c9f3-4e92-80ad-9bf9d8c4b03a
To: /content/Future-Face/archives/part1.tar.gz
100%|██████████| 874M/874M [00:34<00:00, 25.7MB/s]



Downloading: part2.tar.gz


Downloading...
From (original): https://drive.google.com/uc?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b
From (redirected): https://drive.google.com/uc?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&confirm=t&uuid=76f5c033-bf7e-4ae7-80a5-b592e1eea509
To: /content/Future-Face/archives/part2.tar.gz
100%|██████████| 459M/459M [00:20<00:00, 22.2MB/s]



Downloading: part3.tar.gz


Downloading...
From (original): https://drive.google.com/uc?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b
From (redirected): https://drive.google.com/uc?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&confirm=t&uuid=a05af6a7-e4db-4d40-bd55-0c895e990118
To: /content/Future-Face/archives/part3.tar.gz
100%|██████████| 57.0M/57.0M [00:00<00:00, 69.1MB/s]


All archives downloaded.


<h1> 5. Extract all three parts </h1>

In [6]:
# ============================================================
# 5. EXTRACT DATASET
# ============================================================

def extract_tar_gz(archive_path, output_dir):

    print("\n" + "=" * 70)
    print("Extracting:", os.path.basename(archive_path))
    print("=" * 70)

    with tarfile.open(
        archive_path,
        "r:gz"
    ) as tar:

        members = tar.getmembers()

        for member in tqdm(
            members,
            desc=f"Extracting {os.path.basename(archive_path)}"
        ):

            tar.extract(
                member,
                path=output_dir
            )

    print(
        "Finished:",
        os.path.basename(archive_path)
    )


for filename in FILES.keys():

    archive_path = os.path.join(
        ARCHIVE_DIR,
        filename
    )

    extract_tar_gz(
        archive_path,
        DATA_DIR
    )

print("\n" + "=" * 70)
print("ALL ARCHIVES EXTRACTED")
print("=" * 70)


Extracting: part1.tar.gz


Extracting part1.tar.gz:   0%|          | 0/10138 [00:00<?, ?it/s]

/tmp/ipykernel_5298/957663584.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(


Finished: part1.tar.gz

Extracting: part2.tar.gz


Extracting part2.tar.gz:   0%|          | 0/10720 [00:00<?, ?it/s]

Finished: part2.tar.gz

Extracting: part3.tar.gz


Extracting part3.tar.gz:   0%|          | 0/3254 [00:00<?, ?it/s]

Finished: part3.tar.gz

ALL ARCHIVES EXTRACTED


<h1> 6. Find all images and combine them <h1>

In [7]:
# ============================================================
# 6. BUILD ONE COMBINED DATASET
# ============================================================

records = []

for root, dirs, files in os.walk(DATA_DIR):

    for filename in files:

        if not filename.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):
            continue

        try:

            age = int(
                filename.split("_")[0]
            )

        except:

            continue

        image_path = os.path.join(
            root,
            filename
        )

        records.append({
            "path": image_path,
            "age": age
        })


df = pd.DataFrame(records)

print("=" * 70)
print("COMBINED DATASET")
print("=" * 70)

print("Total images:", len(df))
print("Minimum age:", df["age"].min())
print("Maximum age:", df["age"].max())
print("Mean age:", round(df["age"].mean(), 2))
print("Median age:", df["age"].median())

COMBINED DATASET
Total images: 24106
Minimum age: 1
Maximum age: 116
Mean age: 33.05
Median age: 29.0


<h1> 7. Shuffle the entire dataset </h1>

In [8]:
# ============================================================
# 7. RANDOMIZE ENTIRE DATASET
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

df = df.sample(
    frac=1.0,
    random_state=SEED
).reset_index(drop=True)

print("Dataset randomly shuffled.")

print("\nFirst 10 samples:")
print(df.head(10))

Dataset randomly shuffled.

First 10 samples:
                                                path  age
0  /content/Future-Face/UTKFace/part2/51_0_0_2017...   51
1  /content/Future-Face/UTKFace/part2/40_0_1_2017...   40
2  /content/Future-Face/UTKFace/part2/33_1_1_2017...   33
3  /content/Future-Face/UTKFace/part2/35_1_1_2017...   35
4  /content/Future-Face/UTKFace/part1/63_0_0_2017...   63
5  /content/Future-Face/UTKFace/part2/25_0_0_2017...   25
6  /content/Future-Face/UTKFace/part1/10_0_0_2017...   10
7  /content/Future-Face/UTKFace/part2/25_0_0_2017...   25
8  /content/Future-Face/UTKFace/part1/2_1_0_20170...    2
9  /content/Future-Face/UTKFace/part2/26_1_0_2017...   26


<h1> 8. Create age groups for balanced splitting </h1>

In [9]:
# ============================================================
# 8. AGE GROUPS
# ============================================================

def age_group(age):

    if age <= 12:
        return "child"

    elif age <= 17:
        return "teen"

    elif age <= 59:
        return "adult"

    else:
        return "senior"


df["age_group"] = df["age"].apply(
    age_group
)

print(
    df["age_group"].value_counts()
)

age_group
adult     16830
child      3689
senior     2749
teen        838
Name: count, dtype: int64


<h1> 9. Random train / validation / test split </h1>

In [10]:
# ============================================================
# 9. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["age_group"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["age_group"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("=" * 70)
print("DATA SPLIT")
print("=" * 70)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain age groups:")
print(train_df["age_group"].value_counts())

print("\nValidation age groups:")
print(val_df["age_group"].value_counts())

print("\nTest age groups:")
print(test_df["age_group"].value_counts())

DATA SPLIT
Train: 16874
Validation: 3616
Test: 3616

Train age groups:
age_group
adult     11781
child      2582
senior     1924
teen        587
Name: count, dtype: int64

Validation age groups:
age_group
adult     2524
child      554
senior     412
teen       126
Name: count, dtype: int64

Test age groups:
age_group
adult     2525
child      553
senior     413
teen       125
Name: count, dtype: int64


<h1> 10. Save manifests </h1>

In [11]:
# ============================================================
# 10. SAVE DATASET MANIFESTS
# ============================================================

MANIFEST_DIR = os.path.join(
    BASE_DIR,
    "manifests"
)

os.makedirs(
    MANIFEST_DIR,
    exist_ok=True
)

train_df.to_csv(
    os.path.join(
        MANIFEST_DIR,
        "train.csv"
    ),
    index=False
)

val_df.to_csv(
    os.path.join(
        MANIFEST_DIR,
        "validation.csv"
    ),
    index=False
)

test_df.to_csv(
    os.path.join(
        MANIFEST_DIR,
        "test.csv"
    ),
    index=False
)

print("Dataset manifests saved.")

Dataset manifests saved.


<h1> 11. Age-balanced sampler </h1>

In [12]:
# ============================================================
# 11. AGE-BALANCED SAMPLER
# ============================================================

age_counts = (
    train_df["age"]
    .value_counts()
    .sort_index()
)

# Inverse-frequency weights
age_weights = {
    age: 1.0 / count
    for age, count in age_counts.items()
}

sample_weights = np.array([
    age_weights[age]
    for age in train_df["age"]
], dtype=np.float64)

sample_weights = torch.as_tensor(
    sample_weights,
    dtype=torch.double
)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_df),
    replacement=True
)

print(
    "Age-balanced sampler created."
)

print(
    "Samples per epoch:",
    len(train_sampler)
)

print("\nExample age weights:")

for age in [1, 5, 10, 15, 20, 30, 50, 70, 90]:

    if age in age_weights:

        print(
            f"Age {age:3d}: "
            f"{age_weights[age]:.8f}"
        )

Age-balanced sampler created.
Samples per epoch: 16874

Example age weights:
Age   1: 0.00112486
Age   5: 0.00675676
Age  10: 0.00884956
Age  15: 0.00806452
Age  20: 0.00505051
Age  30: 0.00195695
Age  50: 0.00383142
Age  70: 0.00970874
Age  90: 0.01449275


<h1> 12. Dataset and GPU-friendly transforms </h1>

In [13]:
# ============================================================
# 12. DATASET + TRANSFORMS
# ============================================================

IMAGE_SIZE = 224

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=8
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


class AgeDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        image_path = row["path"]
        age = float(row["age"])

        try:

            image = Image.open(
                image_path
            ).convert("RGB")

        except Exception as e:

            raise RuntimeError(
                f"Failed to read image: "
                f"{image_path}"
            ) from e

        if self.transform:

            image = self.transform(
                image
            )

        return (
            image,
            torch.tensor(
                age,
                dtype=torch.float32
            )
        )


train_dataset = AgeDataset(
    train_df,
    train_transform
)

val_dataset = AgeDataset(
    val_df,
    eval_transform
)

test_dataset = AgeDataset(
    test_df,
    eval_transform
)

print("Datasets created.")

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Test:",
    len(test_dataset)
)

Datasets created.
Train: 16874
Validation: 3616
Test: 3616


<h1> 13. GPU-optimized DataLoaders </h1>

In [14]:
# ============================================================
# 13. GPU-OPTIMIZED DATALOADERS
# ============================================================

CPU_COUNT = os.cpu_count()

print("Available CPU threads:", CPU_COUNT)

# Colab commonly gives a limited number of CPUs.
NUM_WORKERS = min(
    8,
    max(2, CPU_COUNT // 2)
)

# Start with 64 for Tesla T4 + ResNet50
BATCH_SIZE = 64

print("Workers:", NUM_WORKERS)
print("Batch size:", BATCH_SIZE)


loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 4
}


train_loader = DataLoader(
    train_dataset,
    sampler=train_sampler,
    shuffle=False,
    drop_last=True,
    **loader_kwargs
)


val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    drop_last=False,
    **loader_kwargs
)


test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    drop_last=False,
    **loader_kwargs
)


print("\nDataLoaders ready.")

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

Available CPU threads: 2
Workers: 2
Batch size: 64

DataLoaders ready.
Train batches: 263
Validation batches: 57
Test batches: 57


<h1> 14. Verify GPU batch transfer </h1>

In [15]:
# ============================================================
# 14. GPU DATA PIPELINE TEST
# ============================================================

images, ages = next(iter(train_loader))

print("CPU batch:")
print("Images:", images.shape)
print("Ages:", ages.shape)
print("Images dtype:", images.dtype)

start = time.time()

images = images.to(
    device,
    non_blocking=True
)

ages = ages.to(
    device,
    non_blocking=True
)

torch.cuda.synchronize()

elapsed = time.time() - start

print("\nGPU batch:")
print("Images device:", images.device)
print("Ages device:", ages.device)

print(
    "GPU transfer time:",
    round(elapsed * 1000, 2),
    "ms"
)

print(
    "\nGPU allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print(
    "GPU reserved:",
    round(
        torch.cuda.memory_reserved() / 1024**3,
        2
    ),
    "GB"
)

CPU batch:
Images: torch.Size([64, 3, 224, 224])
Ages: torch.Size([64])
Images dtype: torch.float32

GPU batch:
Images device: cuda:0
Ages device: cuda:0
GPU transfer time: 137.11 ms

GPU allocated: 0.04 GB
GPU reserved: 0.04 GB


<h1> 15. Create ResNet-50 </h1>

In [16]:
# ============================================================
# 15. RESNET-50 AGE REGRESSION MODEL
# ============================================================

print("=" * 70)
print("CREATING RESNET-50")
print("=" * 70)

weights = ResNet50_Weights.DEFAULT

model = resnet50(
    weights=weights
)

# Replace classification head with age regression
model.fc = nn.Sequential(

    nn.Dropout(
        p=0.25
    ),

    nn.Linear(
        model.fc.in_features,
        1
    )
)

model = model.to(device)

# Channels-last memory format
model = model.to(
    memory_format=torch.channels_last
)

print("Model: ResNet-50")
print("Output: 1 age value")
print("Device:", device)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

CREATING RESNET-50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 149MB/s]


Model: ResNet-50
Output: 1 age value
Device: cuda
Parameters: 23510081


<h1> 16. Loss + optimizer + scheduler </h1>

In [17]:
# ============================================================
# 16. LOSS + OPTIMIZER + SCHEDULER
# ============================================================

criterion = nn.SmoothL1Loss(
    beta=2.0
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
    fused=True
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

print("Loss:", criterion)
print("Optimizer: AdamW")
print("Initial LR:", optimizer.param_groups[0]["lr"])
print("Weight decay:", 1e-4)
print("Scheduler: ReduceLROnPlateau")

Loss: SmoothL1Loss()
Optimizer: AdamW
Initial LR: 0.0001
Weight decay: 0.0001
Scheduler: ReduceLROnPlateau


<h1> 17. Mixed precision setup </h1>

In [18]:
# ============================================================
# 17. AMP
# ============================================================

if device.type == "cuda":

    scaler = torch.amp.GradScaler(
        "cuda"
    )

    print("✅ CUDA AMP enabled")

else:

    scaler = None

    print("AMP disabled")

✅ CUDA AMP enabled




```
# This is formatted as code
```

<h1> 18. One-batch GPU training test </h1>

In [19]:
# ============================================================
# 18. GPU TRAINING TEST
# ============================================================

model.train()

images, ages = next(
    iter(train_loader)
)

images = images.to(
    device,
    non_blocking=True
).contiguous(
    memory_format=torch.channels_last
)

ages = ages.to(
    device,
    non_blocking=True
)

torch.cuda.reset_peak_memory_stats()

optimizer.zero_grad(
    set_to_none=True
)

start_time = time.time()

with torch.amp.autocast(
    device_type="cuda",
    dtype=torch.float16
):

    predictions = model(
        images
    ).squeeze(1)

    loss = criterion(
        predictions,
        ages
    )

scaler.scale(
    loss
).backward()

scaler.step(
    optimizer
)

scaler.update()

torch.cuda.synchronize()

elapsed = time.time() - start_time

print("=" * 70)
print("GPU TRAINING TEST")
print("=" * 70)

print(
    "Loss:",
    loss.item()
)

print(
    "Time:",
    round(elapsed, 3),
    "seconds"
)

print(
    "GPU allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print(
    "GPU peak:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

GPU TRAINING TEST
Loss: 53.889522552490234
Time: 6.825 seconds
GPU allocated: 0.41 GB
GPU peak: 2.88 GB


<h1> 19. Full automatic training </h1>

In [ ]:
# ============================================================
# 19. FULL AUTOMATIC TRAINING
# ============================================================

NUM_EPOCHS = 25

EARLY_STOPPING_PATIENCE = 5

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "future_face_resnet50_best.pth"
)

LAST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "future_face_resnet50_last.pth"
)

HISTORY_PATH = os.path.join(
    CHECKPOINT_DIR,
    "future_face_resnet50_history.json"
)


history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_mae": [],
    "val_rmse": [],
    "learning_rate": [],
    "epoch_time": []
}


best_val_mae = float("inf")

epochs_without_improvement = 0


print("\n")
print("=" * 70)
print("STARTING AUTOMATIC TRAINING")
print("=" * 70)

print("Model: ResNet-50")
print("Epochs:", NUM_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("GPU:", torch.cuda.get_device_name(0))
print("Best model:", BEST_MODEL_PATH)
print("=" * 70)


for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    epoch_start = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    running_loss = 0.0
    sample_count = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Training Epoch {epoch}/{NUM_EPOCHS}",
        leave=True
    )

    for images, ages in train_bar:

        images = images.to(
            device,
            non_blocking=True
        ).contiguous(
            memory_format=torch.channels_last
        )

        ages = ages.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward + loss
        # ----------------------------------------------------

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            predictions = model(
                images
            ).squeeze(1)

            loss = criterion(
                predictions,
                ages
            )

        # ----------------------------------------------------
        # Backprop
        # ----------------------------------------------------

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        batch_size_now = images.size(0)

        running_loss += (
            loss.item()
            * batch_size_now
        )

        sample_count += batch_size_now

        train_loss_now = (
            running_loss
            / sample_count
        )

        train_bar.set_postfix(
            loss=f"{train_loss_now:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.1e}"
        )

    train_loss = (
        running_loss
        / sample_count
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss_total = 0.0
    val_count = 0

    absolute_errors = []
    squared_errors = []

    val_bar = tqdm(
        val_loader,
        desc=f"Validation Epoch {epoch}/{NUM_EPOCHS}",
        leave=True
    )

    with torch.no_grad():

        for images, ages in val_bar:

            images = images.to(
                device,
                non_blocking=True
            ).contiguous(
                memory_format=torch.channels_last
            )

            ages = ages.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                predictions = model(
                    images
                ).squeeze(1)

                loss = criterion(
                    predictions,
                    ages
                )

            batch_size_now = images.size(0)

            val_loss_total += (
                loss.item()
                * batch_size_now
            )

            val_count += batch_size_now

            errors = (
                predictions.float()
                - ages.float()
            )

            absolute_errors.extend(
                torch.abs(errors)
                .cpu()
                .numpy()
            )

            squared_errors.extend(
                (errors ** 2)
                .cpu()
                .numpy()
            )

            val_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

    val_loss = (
        val_loss_total
        / val_count
    )

    val_mae = float(
        np.mean(
            absolute_errors
        )
    )

    val_rmse = float(
        np.sqrt(
            np.mean(
                squared_errors
            )
        )
    )

    # ========================================================
    # SCHEDULER
    # ========================================================

    scheduler.step(
        val_mae
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_time = (
        time.time()
        - epoch_start
    ) / 60.0

    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history["epoch"].append(
        epoch
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_mae"].append(
        val_mae
    )

    history["val_rmse"].append(
        val_rmse
    )

    history["learning_rate"].append(
        current_lr
    )

    history["epoch_time"].append(
        epoch_time
    )

    # Save history continuously
    with open(
        HISTORY_PATH,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n")
    print("-" * 70)
    print(f"Epoch {epoch} complete")
    print("-" * 70)

    print(
        f"Train Loss       : "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation Loss  : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation MAE   : "
        f"{val_mae:.4f} years"
    )

    print(
        f"Validation RMSE  : "
        f"{val_rmse:.4f} years"
    )

    print(
        f"Learning Rate    : "
        f"{current_lr:.8f}"
    )

    print(
        f"Time             : "
        f"{epoch_time:.2f} minutes"
    )

    # ========================================================
    # SAVE LAST CHECKPOINT
    # ========================================================

    torch.save(
        {
            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "train_loss":
                train_loss,

            "val_loss":
                val_loss,

            "val_mae":
                val_mae,

            "val_rmse":
                val_rmse,

            "best_val_mae":
                best_val_mae,

            "history":
                history
        },
        LAST_MODEL_PATH
    )

    print(
        "Last checkpoint saved."
    )

    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_mae < best_val_mae:

        improvement = (
            best_val_mae - val_mae
            if np.isfinite(best_val_mae)
            else 0
        )

        best_val_mae = val_mae

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "scaler_state_dict":
                    scaler.state_dict(),

                "train_loss":
                    train_loss,

                "val_loss":
                    val_loss,

                "val_mae":
                    val_mae,

                "val_rmse":
                    val_rmse,

                "best_val_mae":
                    best_val_mae,

                "history":
                    history
            },
            BEST_MODEL_PATH
        )

        print("\n🔥 NEW BEST MODEL")

        print(
            f"Best Validation MAE: "
            f"{best_val_mae:.4f} years"
        )

        print(
            "Saved:",
            BEST_MODEL_PATH
        )

    else:

        epochs_without_improvement += 1

        print(
            f"\nNo improvement "
            f"({epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE})"
        )

    # ========================================================
    # GPU MEMORY
    # ========================================================

    print(
        "\nGPU allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "GPU reserved:",
        round(
            torch.cuda.memory_reserved()
            / 1024**3,
            2
        ),
        "GB"
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print("\n")
        print("=" * 70)
        print("EARLY STOPPING")
        print("=" * 70)

        print(
            "Best Validation MAE:",
            best_val_mae
        )

        print(
            "Best model:",
            BEST_MODEL_PATH
        )

        break


print("\n")
print("=" * 70)
print("AUTOMATIC TRAINING FINISHED")
print("=" * 70)

print(
    "Best Validation MAE:",
    best_val_mae
)

print(
    "Best checkpoint:",
    BEST_MODEL_PATH
)

print(
    "Last checkpoint:",
    LAST_MODEL_PATH
)



STARTING AUTOMATIC TRAINING
Model: ResNet-50
Epochs: 25
Batch size: 64
GPU: Tesla T4
Best model: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth


Training Epoch 1/25:   0%|          | 0/263 [00:00<?, ?it/s]

Validation Epoch 1/25:   0%|          | 0/57 [00:00<?, ?it/s]



----------------------------------------------------------------------
Epoch 1 complete
----------------------------------------------------------------------
Train Loss       : 19.0166
Validation Loss  : 7.4498
Validation MAE   : 8.4008 years
Validation RMSE  : 11.2103 years
Learning Rate    : 0.00010000
Time             : 4.07 minutes
Last checkpoint saved.

🔥 NEW BEST MODEL
Best Validation MAE: 8.4008 years
Saved: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth

GPU allocated: 0.39 GB
GPU reserved: 0.57 GB


Training Epoch 2/25:   0%|          | 0/263 [00:00<?, ?it/s]

...

Validation Epoch 24/25:   0%|          | 0/57 [00:00<?, ?it/s]



----------------------------------------------------------------------
Epoch 24 complete
----------------------------------------------------------------------
Train Loss       : 1.3737
Validation Loss  : 4.3206
Validation MAE   : 5.2028 years
Validation RMSE  : 7.3974 years
Learning Rate    : 0.00002500
Time             : 3.90 minutes
Last checkpoint saved.

No improvement (1/5)

GPU allocated: 0.39 GB
GPU reserved: 3.45 GB


Training Epoch 25/25:   0%|          | 0/263 [00:00<?, ?it/s]

Validation Epoch 25/25:   0%|          | 0/57 [00:00<?, ?it/s]



----------------------------------------------------------------------
Epoch 25 complete
----------------------------------------------------------------------
Train Loss       : 1.3362
Validation Loss  : 4.2644
Validation MAE   : 5.1434 years
Validation RMSE  : 7.3855 years
Learning Rate    : 0.00002500
Time             : 3.70 minutes
Last checkpoint saved.

🔥 NEW BEST MODEL
Best Validation MAE: 5.1434 years
Saved: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth

GPU allocated: 0.39 GB
GPU reserved: 3.45 GB


AUTOMATIC TRAINING FINISHED
Best Validation MAE: 5.143395900726318
Best checkpoint: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth
Last checkpoint: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_last.pth


<h1> 20. Load the best model

In [ ]:
# ============================================================
# 20. LOAD BEST MODEL
# ============================================================

best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

model.eval()

best_epoch = best_checkpoint[
    "epoch"
]

best_mae = best_checkpoint[
    "val_mae"
]

print("=" * 70)
print("BEST MODEL LOADED")
print("=" * 70)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation MAE:",
    best_mae
)

print(
    "Device:",
    device
)

BEST MODEL LOADED
Best epoch: 25
Best validation MAE: 5.143395900726318
Device: cuda


<h1> 21. Final test evaluation

In [ ]:
# ============================================================
# 21. FINAL TEST EVALUATION
# ============================================================

model.eval()

test_absolute_errors = []
test_squared_errors = []

test_predictions = []
test_actuals = []

test_bar = tqdm(
    test_loader,
    desc="Final Test Evaluation"
)

with torch.no_grad():

    for images, ages in test_bar:

        images = images.to(
            device,
            non_blocking=True
        ).contiguous(
            memory_format=torch.channels_last
        )

        ages = ages.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            predictions = model(
                images
            ).squeeze(1)

        predictions = predictions.float()

        errors = (
            predictions
            - ages
        )

        abs_errors = torch.abs(
            errors
        )

        squared_errors = (
            errors ** 2
        )

        test_absolute_errors.extend(
            abs_errors
            .cpu()
            .numpy()
        )

        test_squared_errors.extend(
            squared_errors
            .cpu()
            .numpy()
        )

        test_predictions.extend(
            predictions
            .cpu()
            .numpy()
        )

        test_actuals.extend(
            ages
            .cpu()
            .numpy()
        )


test_mae = float(
    np.mean(
        test_absolute_errors
    )
)

test_rmse = float(
    np.sqrt(
        np.mean(
            test_squared_errors
        )
    )
)


print("\n")
print("=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(
    f"Test MAE  : "
    f"{test_mae:.4f} years"
)

print(
    f"Test RMSE : "
    f"{test_rmse:.4f} years"
)

print("=" * 70)

Final Test Evaluation:   0%|          | 0/57 [00:00<?, ?it/s]



FINAL TEST RESULTS
Test MAE  : 5.0415 years
Test RMSE : 7.3320 years


<h1> 22. Evaluate performance by age group

In [ ]:
# ============================================================
# 22. AGE-GROUP PERFORMANCE
# ============================================================

results_df = test_df.copy()

results_df["actual_age"] = (
    np.array(test_actuals)
)

results_df["predicted_age"] = (
    np.array(test_predictions)
)

results_df["absolute_error"] = (
    np.abs(
        results_df["actual_age"]
        -
        results_df["predicted_age"]
    )
)

results_df["squared_error"] = (
    (
        results_df["actual_age"]
        -
        results_df["predicted_age"]
    ) ** 2
)


def group_metrics(group):

    return pd.Series({

        "Images":
            len(group),

        "MAE":
            group["absolute_error"].mean(),

        "RMSE":
            np.sqrt(
                group["squared_error"].mean()
            ),

        "Mean Actual Age":
            group["actual_age"].mean(),

        "Mean Predicted Age":
            group["predicted_age"].mean()
    })


group_results = (
    results_df
    .groupby("age_group")
    .apply(
        group_metrics,
        include_groups=False
    )
)

print(
    group_results.round(4)
)

           Images     MAE     RMSE  Mean Actual Age  Mean Predicted Age
age_group                                                              
adult      2525.0  5.1489   6.9490          34.0099             33.6937
child       553.0  2.2742   5.2941           4.0036              5.5767
senior      413.0  8.3416  11.3165          70.7046             64.2409
teen        125.0  4.2118   5.7069          15.3200             16.8838


<h1> 23. Save final test results

In [ ]:
# ============================================================
# 23. SAVE TEST RESULTS
# ============================================================

TEST_RESULTS_PATH = os.path.join(
    CHECKPOINT_DIR,
    "future_face_resnet50_test_results.csv"
)

results_df.to_csv(
    TEST_RESULTS_PATH,
    index=False
)

print(
    "Test predictions saved:"
)

print(
    TEST_RESULTS_PATH
)

Test predictions saved:
/content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_test_results.csv


<h1> 24. Automatically download the final model

In [ ]:
# ============================================================
# 24. AUTOMATICALLY DOWNLOAD FINAL MODEL
# ============================================================

from google.colab import files

FINAL_MODEL_NAME = (
    "Future-Face-ResNet50-Final.pth"
)

FINAL_MODEL_PATH = os.path.join(
    "/content",
    FINAL_MODEL_NAME
)

# Copy best model from Drive to local Colab storage
shutil.copy2(
    BEST_MODEL_PATH,
    FINAL_MODEL_PATH
)

print("=" * 70)
print("FINAL MODEL READY")
print("=" * 70)

print(
    "Model:",
    FINAL_MODEL_PATH
)

print(
    "Validation MAE:",
    best_mae
)

print(
    "Test MAE:",
    test_mae
)

print(
    "Test RMSE:",
    test_rmse
)

print("\nStarting download...")

files.download(
    FINAL_MODEL_PATH
)

FINAL MODEL READY
Model: /content/Future-Face-ResNet50-Final.pth
Validation MAE: 5.143395900726318
Test MAE: 5.041549205780029
Test RMSE: 7.331982612609863

Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<h1> Continue training after 25 epoch

In [30]:
# ============================================================
# FUTURE-FACE RESNET50
# CONTINUE ORIGINAL TRAINING FROM EPOCH 25
# ============================================================

import os
import json
import time
import math
import shutil
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image
from tqdm.auto import tqdm


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_ROOT = "/content/Future-Face/UTKFace"
MANIFEST_DIR = "/content/Future-Face/manifests"

TRAIN_CSV = os.path.join(MANIFEST_DIR, "train.csv")
VAL_CSV   = os.path.join(MANIFEST_DIR, "validation.csv")
TEST_CSV  = os.path.join(MANIFEST_DIR, "test.csv")

CHECKPOINT_DIR = (
    "/content/gdrive/MyDrive/Colab Notebooks/"
    "Dataset/Future-Face/checkpoints"
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "future_face_resnet50_last.pth"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "future_face_resnet50_best.pth"
)

HISTORY_PATH = os.path.join(
    CHECKPOINT_DIR,
    "future_face_resnet50_history.json"
)


# ============================================================
# CONTINUATION SETTINGS
# ============================================================

# Original training was 25 epochs.
# Continue until this epoch.
TARGET_EPOCH = 50

# IMPORTANT:
# Original training used batch size 64.
BATCH_SIZE = 64

NUM_WORKERS = 2

PIN_MEMORY = True
PERSISTENT_WORKERS = True

IMAGE_SIZE = 224

# Keep the original learning rate schedule.
# We restore the scheduler from checkpoint.
INITIAL_LR = 1e-4
WEIGHT_DECAY = 1e-4

# Same loss used during original training.
criterion = nn.SmoothL1Loss()

# Save every epoch.
SAVE_EVERY_EPOCH = True


# ============================================================
# 2. GPU CHECK
# ============================================================

print("=" * 70)
print("FUTURE-FACE RESNET50 CONTINUATION TRAINING")
print("=" * 70)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is NOT available. "
        "Enable GPU runtime before continuing."
    )

device = torch.device("cuda")

print("Device :", device)
print("GPU    :", torch.cuda.get_device_name(0))
print("CUDA   :", torch.version.cuda)
print("PyTorch:", torch.__version__)

gpu_total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU memory total: {gpu_total:.2f} GB")


# ============================================================
# 3. GPU PERFORMANCE SETTINGS
# ============================================================

torch.backends.cudnn.benchmark = True

# Allow TF32 on supported NVIDIA hardware.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("CUDA performance settings enabled.")


# ============================================================
# 4. CHECK PATHS
# ============================================================

print()
print("=" * 70)
print("CHECKING FILES")
print("=" * 70)

required_files = [
    DATASET_ROOT,
    TRAIN_CSV,
    VAL_CSV,
    TEST_CSV,
    LAST_CHECKPOINT,
    BEST_CHECKPOINT,
]

for path in required_files:

    if os.path.exists(path):
        print("✅", path)

    else:
        print("❌ MISSING:", path)


if not os.path.exists(TRAIN_CSV):
    raise FileNotFoundError(TRAIN_CSV)

if not os.path.exists(VAL_CSV):
    raise FileNotFoundError(VAL_CSV)

if not os.path.exists(LAST_CHECKPOINT):
    raise FileNotFoundError(
        "Original last checkpoint was not found:\n"
        + LAST_CHECKPOINT
    )


# ============================================================
# 5. LOAD ORIGINAL MANIFESTS
# ============================================================

print()
print("=" * 70)
print("LOADING ORIGINAL DATA SPLIT")
print("=" * 70)

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Training   :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

print()
print("Training age statistics:")
print(train_df["age"].describe())

print()
print("Validation age statistics:")
print(val_df["age"].describe())

print()
print("Test age statistics:")
print(test_df["age"].describe())


# ============================================================
# 6. DATASET
# ============================================================

class UTKFaceDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["path"]
        age = float(row["age"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        age = torch.tensor(
            age,
            dtype=torch.float32
        )

        return image, age


# ============================================================
# 7. TRANSFORMS
# ============================================================

# IMPORTANT:
# Keep preprocessing compatible with the original model.

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


val_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


# ============================================================
# 8. DATASETS
# ============================================================

train_dataset = UTKFaceDataset(
    train_df,
    train_transform
)

val_dataset = UTKFaceDataset(
    val_df,
    val_transform
)


# ============================================================
# 9. DATALOADERS
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    persistent_workers=PERSISTENT_WORKERS,

    drop_last=False
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    persistent_workers=PERSISTENT_WORKERS,

    drop_last=False
)


print()
print("=" * 70)
print("DATALOADERS")
print("=" * 70)

print("Batch size:", BATCH_SIZE)

print(
    "Training batches   :",
    len(train_loader)
)

print(
    "Validation batches :",
    len(val_loader)
)


# ============================================================
# IMPORTANT SANITY CHECK
# ============================================================

expected_train_batches = math.ceil(
    16874 / 64
)

expected_val_batches = math.ceil(
    3616 / 64
)

print()
print("Expected original configuration:")
print(
    "Training batches   :",
    expected_train_batches
)

print(
    "Validation batches :",
    expected_val_batches
)

if len(train_loader) != expected_train_batches:
    raise RuntimeError(
        "Training loader does not match original "
        "batch configuration."
    )

if len(val_loader) != expected_val_batches:
    raise RuntimeError(
        "Validation loader does not match original "
        "batch configuration."
    )


# ============================================================
# 10. BUILD EXACT RESNET50 ARCHITECTURE
# ============================================================

print()
print("=" * 70)
print("BUILDING RESNET50")
print("=" * 70)

model = models.resnet50(weights=None)

# IMPORTANT:
# Your checkpoint uses fc.1.weight / fc.1.bias.
# Therefore the original model had Sequential FC.
#
# DO NOT replace this with a plain nn.Linear.

model.fc = nn.Sequential(

    nn.Dropout(
        p=0.2
    ),

    nn.Linear(
        model.fc.in_features,
        1
    )
)

model = model.to(device)

# Channels-last
model = model.to(
    memory_format=torch.channels_last
)

print("Architecture : ResNet-50")
print("Output       : Age regression")
print("FC           : Dropout + Linear")
print("Device       :", device)


# ============================================================
# 11. LOSS / OPTIMIZER
# ============================================================

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=INITIAL_LR,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 12. SCHEDULER
# ============================================================

# Same type of scheduler used by the original training.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=2
)


# ============================================================
# 13. AMP
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda"
)


# ============================================================
# 14. LOAD ORIGINAL CHECKPOINT
# ============================================================

print()
print("=" * 70)
print("LOADING ORIGINAL CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    LAST_CHECKPOINT,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded.")

print()
print("Checkpoint keys:")

for key in checkpoint.keys():

    print(" ", key)


start_epoch = checkpoint["epoch"] + 1

print()
print("Checkpoint epoch :", checkpoint["epoch"])
print("Continue from    :", start_epoch)

print(
    "Checkpoint best MAE:",
    checkpoint.get("best_mae")
)

print(
    "Checkpoint val MAE:",
    checkpoint.get("val_mae")
)

print(
    "Checkpoint LR:",
    optimizer.param_groups[0]["lr"]
)


# ============================================================
# 15. LOAD MODEL
# ============================================================

try:

    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    print("✅ Model state loaded successfully.")

except RuntimeError as e:

    print()
    print("❌ MODEL ARCHITECTURE MISMATCH")
    print(e)

    raise


# ============================================================
# 16. LOAD OPTIMIZER
# ============================================================

if "optimizer_state_dict" in checkpoint:

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    print("✅ Optimizer state restored.")


# ============================================================
# 17. LOAD SCHEDULER
# ============================================================

if "scheduler_state_dict" in checkpoint:

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    print("✅ Scheduler state restored.")


# ============================================================
# 18. LOAD AMP SCALER
# ============================================================

if "scaler_state_dict" in checkpoint:

    scaler.load_state_dict(
        checkpoint["scaler_state_dict"]
    )

    print("✅ AMP scaler state restored.")


# ============================================================
# 19. BEST MAE
# ============================================================

best_val_mae = checkpoint.get(
    "best_mae",
    float("inf")
)

print()
print("Best validation MAE so far:")
print(best_val_mae)


# ============================================================
# 20. LOAD TRAINING HISTORY
# ============================================================

if os.path.exists(HISTORY_PATH):

    with open(
        HISTORY_PATH,
        "r"
    ) as f:

        history = json.load(f)

    print()
    print("✅ Existing history loaded.")

else:

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_mae": [],
        "val_rmse": [],
        "learning_rate": [],
        "epoch_time": []
    }

    print()
    print("⚠️ History file not found.")
    print("Creating new history container.")


# ============================================================
# 21. VALIDATE CHECKPOINT BEFORE TRAINING
# ============================================================

print()
print("=" * 70)
print("CHECKPOINT SANITY CHECK")
print("=" * 70)

model.eval()

with torch.no_grad():

    images, ages = next(
        iter(val_loader)
    )

    images = images.to(
        device,
        non_blocking=True
    ).contiguous(
        memory_format=torch.channels_last
    )

    ages = ages.to(
        device,
        non_blocking=True
    )

    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        predictions = model(
            images
        ).squeeze(1)

    print("Actual ages:")
    print(
        ages[:10].detach().cpu().numpy()
    )

    print()
    print("Predicted ages:")
    print(
        predictions[:10].float().detach().cpu().numpy()
    )


# ============================================================
# 22. CONTINUATION TRAINING
# ============================================================

print()
print("=" * 70)
print("STARTING CONTINUATION TRAINING")
print("=" * 70)

print(
    f"Starting epoch : {start_epoch}"
)

print(
    f"Target epoch   : {TARGET_EPOCH}"
)

print(
    f"Batch size     : {BATCH_SIZE}"
)

print(
    f"GPU            : {torch.cuda.get_device_name(0)}"
)

print("=" * 70)


for epoch in range(
    start_epoch,
    TARGET_EPOCH + 1
):

    epoch_start = time.time()

    print()
    print("=" * 70)
    print(
        f"EPOCH {epoch}/{TARGET_EPOCH}"
    )
    print("=" * 70)


    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    running_loss = 0.0
    sample_count = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Training Epoch {epoch}",
        leave=True
    )

    for images, ages in train_bar:

        images = images.to(
            device,
            non_blocking=True
        ).contiguous(
            memory_format=torch.channels_last
        )

        ages = ages.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            predictions = model(
                images
            ).squeeze(1)

            loss = criterion(
                predictions,
                ages
            )


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()


        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        current_batch_size = images.size(0)

        running_loss += (
            loss.item()
            * current_batch_size
        )

        sample_count += current_batch_size

        train_bar.set_postfix({

            "loss":
            f"{loss.item():.4f}",

            "lr":
            f"{optimizer.param_groups[0]['lr']:.2e}"
        })


    train_loss = (
        running_loss
        / sample_count
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss_total = 0.0

    absolute_errors = []
    squared_errors = []

    val_samples = 0


    val_bar = tqdm(
        val_loader,
        desc=f"Validation Epoch {epoch}",
        leave=True
    )


    with torch.no_grad():

        for images, ages in val_bar:

            images = images.to(
                device,
                non_blocking=True
            ).contiguous(
                memory_format=torch.channels_last
            )

            ages = ages.to(
                device,
                non_blocking=True
            )


            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                predictions = model(
                    images
                ).squeeze(1)

                loss = criterion(
                    predictions,
                    ages
                )


            batch_size_now = images.size(0)

            val_loss_total += (
                loss.item()
                * batch_size_now
            )

            val_samples += batch_size_now


            errors = (
                predictions.float()
                - ages.float()
            )

            absolute_errors.append(
                errors.abs().cpu()
            )

            squared_errors.append(
                (errors ** 2).cpu()
            )


    val_loss = (
        val_loss_total
        / val_samples
    )


    all_absolute_errors = torch.cat(
        absolute_errors
    )

    all_squared_errors = torch.cat(
        squared_errors
    )


    val_mae = (
        all_absolute_errors.mean().item()
    )

    val_rmse = math.sqrt(
        all_squared_errors.mean().item()
    )


    # ========================================================
    # SCHEDULER
    # ========================================================

    old_lr = optimizer.param_groups[0]["lr"]

    scheduler.step(
        val_mae
    )

    new_lr = optimizer.param_groups[0]["lr"]


    # ========================================================
    # TIME
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print()
    print("-" * 70)

    print(
        f"Epoch {epoch} complete"
    )

    print(
        f"Train Loss       : {train_loss:.4f}"
    )

    print(
        f"Validation Loss  : {val_loss:.4f}"
    )

    print(
        f"Validation MAE   : {val_mae:.4f} years"
    )

    print(
        f"Validation RMSE  : {val_rmse:.4f} years"
    )

    print(
        f"Learning Rate    : {new_lr:.8f}"
    )

    print(
        f"Time             : {epoch_time / 60:.2f} minutes"
    )

    print("-" * 70)


    # ========================================================
    # UPDATE HISTORY
    # ========================================================

    history["epoch"].append(
        epoch
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_mae"].append(
        val_mae
    )

    history["val_rmse"].append(
        val_rmse
    )

    history["learning_rate"].append(
        new_lr
    )

    history["epoch_time"].append(
        epoch_time
    )


    # ========================================================
    # SAVE LAST CHECKPOINT
    # ========================================================

    last_checkpoint = {

        "epoch":
        epoch,

        "model_state_dict":
        model.state_dict(),

        "optimizer_state_dict":
        optimizer.state_dict(),

        "scheduler_state_dict":
        scheduler.state_dict(),

        "scaler_state_dict":
        scaler.state_dict(),

        "best_mae":
        best_val_mae,

        "val_mae":
        val_mae,

        "val_rmse":
        val_rmse,

        "val_loss":
        val_loss,

        "train_loss":
        train_loss,

        "batch_size":
        BATCH_SIZE,

        "architecture":
        "ResNet50",

        "description":
        "Future-Face ResNet50 continuation from original epoch 25"
    }


    torch.save(
        last_checkpoint,
        LAST_CHECKPOINT
    )

    print(
        "✅ Last checkpoint saved."
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_mae < best_val_mae:

        improvement = (
            best_val_mae
            - val_mae
        )

        best_val_mae = val_mae

        best_checkpoint = {

            "epoch":
            epoch,

            "model_state_dict":
            model.state_dict(),

            "optimizer_state_dict":
            optimizer.state_dict(),

            "scheduler_state_dict":
            scheduler.state_dict(),

            "scaler_state_dict":
            scaler.state_dict(),

            "best_mae":
            best_val_mae,

            "val_mae":
            val_mae,

            "val_rmse":
            val_rmse,

            "val_loss":
            val_loss,

            "train_loss":
            train_loss,

            "batch_size":
            BATCH_SIZE,

            "architecture":
            "ResNet50",

            "description":
            "Future-Face ResNet50 best model"
        }


        torch.save(
            best_checkpoint,
            BEST_CHECKPOINT
        )

        print()
        print("🔥 NEW BEST MODEL")

        print(
            f"Improvement     : "
            f"{improvement:.4f} years"
        )

        print(
            f"Best Validation MAE: "
            f"{best_val_mae:.4f} years"
        )

        print(
            "Saved:",
            BEST_CHECKPOINT
        )

    else:

        print(
            f"No improvement."
            f" Best MAE remains "
            f"{best_val_mae:.4f}"
        )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    with open(
        HISTORY_PATH,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    # ========================================================
    # GPU MEMORY
    # ========================================================

    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )

    print()
    print(
        f"GPU allocated: "
        f"{allocated:.2f} GB"
    )

    print(
        f"GPU reserved : "
        f"{reserved:.2f} GB"
    )


# ============================================================
# 23. TRAINING FINISHED
# ============================================================

print()
print("=" * 70)
print("CONTINUATION TRAINING FINISHED")
print("=" * 70)

print(
    "Last checkpoint:"
)

print(
    LAST_CHECKPOINT
)

print()

print(
    "Best checkpoint:"
)

print(
    BEST_CHECKPOINT
)

print()

print(
    f"Best Validation MAE: "
    f"{best_val_mae:.4f} years"
)

print("=" * 70)

FUTURE-FACE RESNET50 CONTINUATION TRAINING
Device : cuda
GPU    : Tesla T4
CUDA   : 13.0
PyTorch: 2.11.0+cu130
GPU memory total: 14.56 GB
CUDA performance settings enabled.

CHECKING FILES
✅ /content/Future-Face/UTKFace
✅ /content/Future-Face/manifests/train.csv
✅ /content/Future-Face/manifests/validation.csv
✅ /content/Future-Face/manifests/test.csv
✅ /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_last.pth
✅ /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth

LOADING ORIGINAL DATA SPLIT
Training   : 16874
Validation : 3616
Test       : 3616

Training age statistics:
count    16874.000000
mean        33.072893
std         20.136458
min          1.000000
25%         23.000000
50%         29.000000
75%         45.000000
max        116.000000
Name: age, dtype: float64

Validation age statistics:
count    3616.000000
mean       33.120852
std        20.280578
min         1.000000
25%        23.0

Training Epoch 29:   0%|          | 0/264 [00:00<?, ?it/s]

Validation Epoch 29:   0%|          | 0/57 [00:00<?, ?it/s]


----------------------------------------------------------------------
Epoch 29 complete
Train Loss       : 2.8373
Validation Loss  : 2.9867
Validation MAE   : 3.4402 years
Validation RMSE  : 5.3748 years
Learning Rate    : 0.00002500
Time             : 3.36 minutes
----------------------------------------------------------------------
✅ Last checkpoint saved.

🔥 NEW BEST MODEL
Improvement     : inf years
Best Validation MAE: 3.4402 years
Saved: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth

GPU allocated: 1.64 GB
GPU reserved : 2.35 GB

EPOCH 30/50


Training Epoch 30:   0%|          | 0/264 [00:00<?, ?it/s]

...

Validation Epoch 49:   0%|          | 0/57 [00:00<?, ?it/s]


----------------------------------------------------------------------
Epoch 49 complete
Train Loss       : 1.3252
Validation Loss  : 2.7056
Validation MAE   : 3.1510 years
Validation RMSE  : 5.0726 years
Learning Rate    : 0.00000100
Time             : 3.43 minutes
----------------------------------------------------------------------
✅ Last checkpoint saved.
No improvement. Best MAE remains 3.1307

GPU allocated: 1.28 GB
GPU reserved : 4.72 GB

EPOCH 50/50


Training Epoch 50:   0%|          | 0/264 [00:00<?, ?it/s]

Validation Epoch 50:   0%|          | 0/57 [00:00<?, ?it/s]


----------------------------------------------------------------------
Epoch 50 complete
Train Loss       : 1.3137
Validation Loss  : 2.6952
Validation MAE   : 3.1401 years
Validation RMSE  : 5.0858 years
Learning Rate    : 0.00000100
Time             : 3.40 minutes
----------------------------------------------------------------------
✅ Last checkpoint saved.
No improvement. Best MAE remains 3.1307

GPU allocated: 1.29 GB
GPU reserved : 4.72 GB

CONTINUATION TRAINING FINISHED
Last checkpoint:
/content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_last.pth

Best checkpoint:
/content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth

Best Validation MAE: 3.1307 years


<h1> Download New Model

In [31]:
from google.colab import files
import os

best_model = "/content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth"

if not os.path.exists(best_model):
    raise FileNotFoundError(f"Model not found: {best_model}")

print("=" * 60)
print("DOWNLOADING BEST FUTURE-FACE MODEL")
print("=" * 60)

print(f"Model: {best_model}")
print(f"Size: {os.path.getsize(best_model) / (1024**2):.2f} MB")

files.download(best_model)

DOWNLOADING BEST FUTURE-FACE MODEL
Model: /content/gdrive/MyDrive/Colab Notebooks/Dataset/Future-Face/checkpoints/future_face_resnet50_best.pth
Size: 269.50 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>